# Tutorial for Spectral Extraction Module

2025-07-21: Algorithms are being actively developed on the spectral-extraction branch. Code is in the module `spectral_extraction_gjg`. Eventually this code will replace the current DRP implementation of optimal extraction in the module `spectral_extraction`, but for now both versions exist simultaneously in the codebase.

The primary interface for using the spectral extraction module is the class SpectralExtractionAlg
The primary methods of this class are:
1. extract_orderlet() to extract the 1D spectrum for a single orderlet
2. extract_ccd() to extract the 1D spectra for all orders/orderlets on either the GREEN or RED ccd

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import warnings

from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import median_filter
from scipy.interpolate import LSQUnivariateSpline
from astropy.io import fits
from astropy.stats import mad_std
from numpy.polynomial import polynomial as poly

from kpfpipe.models.level0 import KPF0
from kpfpipe.models.level1 import KPF1
from modules.Utils.kpf_parse import get_datecode

%matplotlib inline

## Data Inputs

The files needed to perform spectral extraction are
- **target_2D** science frame, a KPF0 level 2D object
- **master_flat_2D** master flat, a KPF0 level 2D object
- **order_trace_green** and **order_trace_red**, paths to the order trace csv files on disk
- **start_order_green** and **start_order_red**, integers indicating the starting order trace index
- a **config file** path must also be supplied

In order to perform stray light removal, you will also need **order_mask_2D**, a KPF0 level 2D master order mask

In [ ]:
OBS_ID = 'KP.20240405.49597.71'    # TOI-1184 (K Star) | V = 11.0
#OBS_ID = 'KP.20241022.48910.02'    # HR 3662 (A star) | V = 4.8

datecode = get_datecode(OBS_ID)

filepath = os.path.join('/data/2D/', datecode, f'{OBS_ID}_2D.fits')
target_2D = KPF0.from_fits(filepath, data_type='KPF')

filepath = os.path.join('/data/masters/', datecode, f'kpf_{datecode}_master_flat.fits')
master_flat_2D = KPF0.from_fits(filepath, data_type='KPF')

filepath = os.path.join('/data/masters/', datecode, f'kpf_{datecode}_order_mask.fits')
master_order_mask = KPF0.from_fits(filepath, data_type='KPF')

order_trace_green = os.path.join('/data/masters/', datecode, f'kpf_{datecode}_master_flat_GREEN_CCD.csv')
order_trace_red = os.path.join('/data/masters/', datecode, f'kpf_{datecode}_master_flat_RED_CCD.csv')

# lookup in caldates/start_order.csv
start_order_green = 0
start_order_red = 1

## Stray Light Estimation

Before running spectral extraction, we need to estimate and remove the stray light background. Details for doing so can be found at the [Stray Light Tutorial](https://kpf-pipeline.readthedocs.io/en/latest/tutorials/Stray_Light_Tutorial.html).

In [ ]:
from modules.stray_light.src.alg import StrayLightAlg

stray_light_modeler = StrayLightAlg(target_2D, master_order_mask, '/code/KPF-Pipeline/modules/stray_light/configs/default.cfg')

for chip in ['GREEN', 'RED']:
    print(f'{chip} CCD')
    stray_light_modeler.target_2D = stray_light_modeler.remove_stray_light(chip, 
                                                                           method='polynomial', 
                                                                           polyorder=5, 
                                                                           regularize='auto',
                                                                           edge_clip=256,
                                                                           mask_buffer=2
                                                                          )

target_2D = stray_light_modeler.target_2D

## Spectral Extraction

To initialize the SpectralExtractionAlg, pass the data inputs enumerated above in the order indicated below

In [ ]:
from modules.spectral_extraction_gjg.src.alg import SpectralExtractionAlg

spectrum_extractor = SpectralExtractionAlg(target_2D, 
                                           master_flat_2D,
                                           order_trace_green, 
                                           order_trace_red, 
                                           start_order_green,
                                           start_order_red,
                                           '/code/KPF-Pipeline/modules/spectral_extraction_gjg/configs/default.cfg'
                                          )

### Extracting a single orderlet

To extract a single orderlet, use the extract_orderlet() method. At the moment, you'll need to manually determine what trace_index corresponds to the orderlet you want. Future versions will include support for specifying an order number and fiber.

Required positional arguments:
- **chip** = *'GREEN'* or *'RED'*
- **trace_index** = *int*

Optional keyword argmuments have their default values specified in the config file. The kwarg **max_iter**=*int* sets the maximum number of iterations the routine will run. All other kwargs relate to the parameters of the optimal extraction algorithm. See the class documentation and [Horne 1986](https://ui.adsabs.harvard.edu/abs/1986PASP...98..609H/abstract) for details.

In [ ]:
spectrum_1D_opt, variance_1D_opt, profile_2D, mask_2D = spectrum_extractor.extract_orderlet('RED', 
                                                                                            22, 
                                                                                            method='optimal',
                                                                                            max_iter=20,
                                                                                            profile_filter_size=101,
                                                                                            profile_num_knots=32,
                                                                                            profile_sigma_clip=3.0,
                                                                                            extraction_sigma_clip=5.0
                                                                                           )

The method returns
- spectrum_1D: ndarray, 1D extracted spectrum
- variance_1D: ndarray, 1D extracted variance
- profile_2D: ndarray, 2D spatial profile estimated from master flat
- mask_2D: ndarray, 2D boolean mask flagging bad pixels and out-of-trace pixels

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,6))

ax[0].plot(spectrum_1D_opt, c='C0', label='flux')
ax[0].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[0].legend(fontsize=16)
ax[1].plot(variance_1D_opt, c='C1', label='variance')
ax[1].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[1].legend(fontsize=16)
ax[1].set_xlabel("Pixel column", fontsize=16)

plt.show()

In [ ]:
fig, ax = plt.subplots(8,2, figsize=(16,8))

for i in range(8):
    ax[i,0].imshow(profile_2D[:,i*500:(i+1)*500], 
                 origin='lower', 
                 vmin=np.nanpercentile(profile_2D, 0.5), 
                 vmax=np.nanpercentile(profile_2D, 99.5)
                )
    ax[i,0].set_xticks(np.arange(0,501,100), np.arange(i*500,(i+1)*500+1,100), fontsize=8)
    ax[i,0].set_yticks([])
    
    ax[i,1].imshow(mask_2D[:,i*500:(i+1)*500], origin='lower')
    ax[i,1].set_xticks(np.arange(0,501,100), np.arange(i*500,(i+1)*500+1,100), fontsize=8)
    ax[i,1].set_yticks([])
    
ax[0,0].set_title("Profile", fontsize=20)
ax[0,1].set_title("Mask", fontsize=20)

plt.tight_layout()
plt.show()

For comparison, let's look at the results from standard box extraction. Note that the box extraction method does not estimate the spatial profile and returns **None** for both the profile and mask.

In [ ]:
spectrum_1D_box, variance_1D_box, _, _ = spectrum_extractor.extract_orderlet('RED', 22, method='box')

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,6))

ax[0].plot(spectrum_1D_box, c='C0', label='flux')
ax[0].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[0].legend(fontsize=16)
ax[1].plot(variance_1D_box, c='C1', label='variance')
ax[1].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[1].legend(fontsize=16)
ax[1].set_xlabel("Pixel column", fontsize=16)

plt.show()

The most obvious difference is the lack of outlier removal in the box extraction algorithm. Let's look at a couple of other views of the data.

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,6))

snr_box = spectrum_1D_box/np.sqrt(variance_1D_box)
snr_opt = spectrum_1D_opt/np.sqrt(variance_1D_opt)

ax[0].plot(snr_box, c='grey', label='box extraction S/N')
ax[0].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[0].legend(fontsize=16, loc='upper left')
ax[1].plot(snr_opt, c='k', label='optimal extraction S/N')
ax[1].set_ylabel("Counts ($e^-$)", fontsize=16)
ax[1].legend(fontsize=16, loc='upper left')
ax[1].set_xlabel("Pixel column", fontsize=16)

plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,6))

ax[0].plot(spectrum_1D_opt-spectrum_1D_box, c='grey', label='optimal - box')
ax[0].legend(fontsize=16, loc='upper left')

ax[1].plot(spectrum_1D_opt/spectrum_1D_box, c='k', label='optimal / box')
ax[1].legend(fontsize=16, loc='upper left')

plt.show()

### Extracting a full CCD

To extract all spectra from either the GREEN or RED ccd, use the extract_ccd() method to produce a new KPF L1 object. The interface is identical to that of extract_orderlet(), except that you will no longer specify a single orderlet to extract.

When SpectralExtraction class is initialized, it creates a KPF L1 object from the supplied KPF 2D object. The code snippet below will produce an attribute target_l1 that contains extracted FLUX and VAR for all five fibers and every order on both the RED and GREEN ccds.

In [ ]:
for chip in ['GREEN', 'RED']:
    print(f'{chip} CCD')
    spectrum_extractor.target_l1 = spectrum_extractor.extract_ccd(chip, 
                                                                  method='optimal',
                                                                  max_iter=20,
                                                                  profile_filter_size=101,
                                                                  profile_num_knots=32,
                                                                  profile_sigma_clip=3.0,
                                                                  extraction_sigma_clip=5.0
                                                                 )

In [ ]:
L1 = spectrum_extractor.target_l1
ext = 'SCI_FLUX2'

i = 0
for chip in ['GREEN', 'RED']:
    norder, ncol = L1[f'{chip}_{ext}'].shape
    
    for o in range(norder):
        i += 1
        
        plt.figure(figsize=(20,4))
        plt.plot(L1[f'{chip}_{ext}'][o], color=cm.jet(i/67))
        plt.title(f'{ext} | Order {i}')
        plt.xlabel("Pixel column", fontsize=16)
        plt.ylabel("Counts ($e^-$)", fontsize=16)
        plt.gca().invert_xaxis()
        plt.show()

## Compare to current DRP

Let's read in an existing L1 file and compare to the current extraction in the DRP

In [ ]:
opt_l1 = spectrum_extractor.target_l1
drp_l1 = KPF1.from_fits(os.path.join('/data/L1/', datecode, f'{OBS_ID}_L1.fits'), data_type='KPF')

ext = 'SCI_FLUX2'

i = 0
for chip in ['GREEN', 'RED']:
    norder, ncol = opt_l1[f'{chip}_{ext}'].shape
    
    for o in range(norder):
        i += 1

        fig, ax = plt.subplots(2,1, figsize=(20,6))
        
        ax[0].plot(opt_l1[f'{chip}_{ext}'][o], c=cm.jet(i/67), label='optimal')
        ax[0].legend(fontsize=16, loc='upper left')
        ax[0].set_ylabel("Counts ($e^-$)", fontsize=16)
        ax[0].set_title(f'{ext} | Order {i}', fontsize=20)
        ax[0].invert_xaxis()
        
        ax[1].plot(drp_l1[f'{chip}_{ext}'][o], c=cm.jet(i/67), label='current DRP')
        ax[1].legend(fontsize=16, loc='upper left')
        ax[1].set_xlabel("Pixel column", fontsize=16)
        ax[1].set_ylabel("Counts ($e^-$)", fontsize=16)
        ax[1].invert_xaxis()
        
        plt.show()

In [ ]:
opt_l1 = spectrum_extractor.target_l1
drp_l1 = KPF1.from_fits(os.path.join('/data/L1/', datecode, f'{OBS_ID}_L1.fits'), data_type='KPF')

ext = 'SCI_FLUX2'

i = 0
for chip in ['GREEN', 'RED']:
    norder, ncol = opt_l1[f'{chip}_{ext}'].shape
    
    for o in range(norder):
        i += 1

        print(np.sum(drp_l1[f'{chip}_{ext}'][o] < 0))

        fig, ax = plt.subplots(2,1, figsize=(20,6))
        
        ax[0].plot(opt_l1[f'{chip}_{ext}'][o] - drp_l1[f'{chip}_{ext}'][o], c=cm.jet(i/67), label='optimal - DRP')
        ax[0].legend(fontsize=16, loc='upper left')
        ax[0].set_ylabel("Counts ($e^-$)", fontsize=16)
        ax[0].set_title(f'{ext} | Order {i}', fontsize=20)
        ax[0].invert_xaxis()
        
        ax[1].plot(opt_l1[f'{chip}_{ext}'][o] / drp_l1[f'{chip}_{ext}'][o], c=cm.jet(i/67), label='optimal / DRP')
        ax[1].legend(fontsize=16, loc='upper left')
        ax[1].set_xlabel("Pixel column", fontsize=16)
        ax[1].set_ylabel("Counts ($e^-$)", fontsize=16)
        ax[1].invert_xaxis()
        
        plt.show()